In [7]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os

### Firstly we create a master data from joining each clean data

In [3]:
user  = pd.read_csv("../data/clean/user_clean.csv", parse_dates=["signup_date"])
act    = pd.read_csv("../data/clean/activity_clean.csv", parse_dates=["event_date"])
trx    = pd.read_csv("../data/clean/transactions_clean.csv", parse_dates=["transaction_date"])
status = pd.read_csv("../data/clean/status_clean.csv")

In [44]:
user.head()

,user_id,signup_date,acquisition_channel,age,city,age_missing
0,1,2024-01-01,Referral,33.0,Jakarta,0
1,2,2024-01-02,Affiliate,23.0,Bandung,0
2,3,2024-01-03,Organic,26.0,Jakarta,0
3,4,2024-01-04,Referral,28.0,Bandung,0
4,5,2024-01-05,Referral,47.0,Bandung,0


In [45]:
act.head()

,Unnamed: 0,user_id,event_date,session_count,feature_used
0,0,1,2025-03-26,1,payment
1,1,1,2025-01-01,3,loan_apply
2,2,1,2025-03-18,4,payment
3,3,1,2025-03-06,1,promo_click
4,4,1,2025-02-01,2,promo_click


In [58]:
trx.head()

,user_id,transaction_date,amount,invalid_amount,amount_clean
0,1,2025-01-30,4967920,0,4967920.0
1,1,2025-03-22,4660351,0,4660351.0
2,1,2025-02-01,4813094,0,4813094.0
3,1,2025-04-20,3395453,0,3395453.0
4,1,2025-03-11,5970747,0,5970747.0


In [56]:
print(f"total transaction amount: {trx.amount_clean.sum()}")
print(f"average transaction amount: {trx.amount_clean.mean()}")
print(f"range data from {trx.transaction_date.min()} until {trx.transaction_date.max()}")

total transaction amount: 3549217582.0
average transaction amount: 3500214.5779092703
range data from 2025-01-01 00:00:00 until 2025-04-30 00:00:00


In [47]:
status.head()

,Unnamed: 0,user_id,loan_count,avg_transaction_amount,is_churn,churn_date
0,0,1,7,4.252810e+06,0,NaN
1,1,2,6,3.331625e+06,0,NaN
2,2,3,6,3.574123e+06,0,NaN
3,3,4,2,2.630618e+06,0,NaN
4,4,5,5,4.629785e+06,0,NaN


In [54]:
print(f"total transaction volume: {status.loan_count.sum()}")
print(f"average transaction amount: {status.avg_transaction_amount.mean()}")

total transaction volume: 1014
average transaction amount: 3523077.056504762


In [4]:
# we define first snapshot date for aggregation purpose
snapshot_date = act.event_date.max()

In [6]:
# Transaction feature (per user)
t = trx.groupby("user_id").agg(
    n_trx=("amount_clean", "size"),                  # total transaction
    total_amount=("amount_clean", "sum"),
    avg_amount=("amount_clean", "mean"),
    first_trx=("transaction_date", "min"),
    last_trx=("transaction_date", "max"))
t["trx_span_days"]    = (t.last_trx - t.first_trx).dt.days       # long distance transaction
t["trx_recency_days"] = (snapshot_date - t.last_trx).dt.days               # days since last transaction
t = t.drop(columns=["first_trx", "last_trx"])

# Activity features (per user)
a = act.groupby("user_id").agg(
    total_sessions=("session_count", "sum"),
    active_days=("event_date", "nunique"),
    last_event=("event_date", "max"))
a["recency_days"] = (snapshot_date - a.last_event).dt.days
a = a.drop(columns=["last_event"])

# Application feature usage: % of events per feature type
fu = pd.crosstab(act.user_id, act.feature_used, normalize="index") * 100
fu.columns = ["pct_" + c for c in fu.columns]

# Combine all (1 row = 1 user)
df = user.set_index("user_id")
df["tenure_days"] = (snapshot_date - df.signup_date).dt.days   # 
df = df.drop(columns=["signup_date", "age_missing"])
df = df.join(t).join(a).join(fu).join(status.set_index("user_id")[["is_churn"]])

df.to_csv("../data/clean/user_features.csv")
print(df.shape)
print(df.head(3).T)

(250, 17)
user_id                           1               2               3
acquisition_channel        Referral       Affiliate         Organic
age                            33.0            23.0            26.0
city                        Jakarta         Bandung         Jakarta
tenure_days                     485             484             483
n_trx                             7               6               6
total_amount             29769669.0      19989748.0      21444736.0
avg_amount           4252809.857143  3331624.666667  3574122.666667
trx_span_days                    85              68              84
trx_recency_days                  5              18              13
total_sessions                   11              35              46
active_days                       5              11              15
recency_days                     35              26               8
pct_browse                      0.0            25.0            37.5
pct_loan_apply                 20.0   

In [59]:
df

,acquisition_channel,age,city,tenure_days,n_trx,total_amount,avg_amount,trx_span_days,trx_recency_days,total_sessions,active_days,recency_days,pct_browse,pct_loan_apply,pct_payment,pct_promo_click,is_churn,status,age_group,segment
user_id,,,,,,,,,,,,,,,,,,,,
1,Referral,33.0,Jakarta,485,7,29769669.0,4.252810e+06,85,5,11,5,35,0.000000,20.000000,40.000000,40.000000,0,Active,30-39,Low risk\n(2+ transaction)
2,Affiliate,23.0,Bandung,484,6,19989748.0,3.331625e+06,68,18,35,11,26,25.000000,33.333333,25.000000,16.666667,0,Active,<30,Low risk\n(2+ transaction)
3,Organic,26.0,Jakarta,483,6,21444736.0,3.574123e+06,84,13,46,15,8,37.500000,25.000000,18.750000,18.750000,0,Active,<30,Low risk\n(2+ transaction)
4,Referral,28.0,Bandung,482,2,5261235.0,2.630618e+06,81,4,49,16,5,29.411765,29.411765,29.411765,11.764706,0,Active,<30,Low risk\n(2+ transaction)
5,Referral,47.0,Bandung,481,5,23148924.0,4.629785e+06,98,14,15,7,5,28.571429,42.857143,28.571429,0.000000,0,Active,40-49,Low risk\n(2+ transaction)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,Ads,33.0,Jakarta,240,2,6603591.0,3.301796e+06,77,32,44,14,16,21.428571,21.428571,28.571429,28.571429,0,Active,30-39,Low risk\n(2+ transaction)
247,Organic,32.0,Jakarta,239,1,1134780.0,1.134780e+06,0,9,34,12,20,25.000000,25.000000,25.000000,25.000000,1,Churn,30-39,High risk\n(only 1 transaction)
248,Organic,51.0,Jakarta,238,7,19047053.0,2.721008e+06,99,7,59,17,2,27.777778,16.666667,27.777778,27.777778,0,Active,50+,Low risk\n(2+ transaction)


In [62]:
print(df.total_amount.sum())
print(df.total_amount.mean())

3549217582.0
14196870.328


### EDA

In [8]:
# Graphic style

plt.rcParams.update({
    "font.family": ["Calibri", "DejaVu Sans"], "font.size": 12,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False})
RED, BLUE, GREY = "#C0392B", "#2E86AB", "#95A5A6"
os.makedirs("charts", exist_ok=True)

In [32]:
df = pd.read_csv("../data/clean/user_features.csv", index_col="user_id")
df["status"] = df.is_churn.map({0: "Active", 1: "Churn"})
avg_churn = df.is_churn.mean() * 100
print(f"Total user: {len(df)} | Churn rate: {avg_churn:.1f}%")

def compare(ax, col, title, scale=1, fmt="{:.1f}"):
    """Bar average active vs Churn for one metric."""
    m = df.groupby("status")[col].mean().reindex(["Active", "Churn"]) / scale
    bars = ax.bar(m.index, m.values, color=[BLUE, RED], width=0.55)
    for b, v in zip(bars, m.values):
        ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v), ha="center", va="bottom", fontweight="bold")
    ax.set_title(title); ax.set_ylim(0, m.max() * 1.2)
    return m

def rate_bars(ax, groups, title, order=None):
    """Bar churn rate (%) per group + average line."""
    r = df.groupby(groups, observed=True).is_churn.mean() * 100
    if order: r = r.reindex(order)
    bars = ax.bar(r.index.astype(str), r.values, color=GREY, width=0.6)
    for b, v in zip(bars, r.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}%", ha="center", va="bottom", fontweight="bold")
    ax.axhline(avg_churn, ls="--", c="black", lw=1)
    ax.set_title(title); ax.set_ylim(0, 100); ax.set_ylabel("Churn rate (%)")
    return r


Total user: 250 | Churn rate: 29.6%


In [33]:
# 1. Churn rate per total loan

fig, ax = plt.subplots(figsize=(10, 5))
r = df.groupby("n_trx").is_churn.mean() * 100
bars = ax.bar(r.index.astype(str), r.values, color=[RED if k == 1 else GREY for k in r.index], width=0.65)
for b, v in zip(bars, r.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}%", ha="center", va="bottom", fontweight="bold")
ax.axhline(avg_churn, ls="--", c="black", lw=1); ax.text(6.45, avg_churn + 1.5, f"Average {avg_churn:.0f}%", ha="left")
ax.set_xlabel("Total number of loans taken out by the user"); ax.set_ylabel("Churn rate (%)"); ax.set_ylim(0, 80)
ax.set_title("Users with only 1 loan result churn 69%, about 3x times higher than users who borrows again")
plt.tight_layout(); plt.savefig("charts/1_churn_per_jumlah_pinjaman.png", dpi=150); plt.close()

In [71]:
# 2. Transaction behavior: Churn vs Non-churn

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
print("\n[Transaction] average:")
print(df.groupby("status")[["n_trx", "trx_span_days", "total_amount"]].mean().round(0))
compare(axs[0], "n_trx", "Average number of transaction")
compare(axs[1], "trx_recency_days", "Average of last transaction days", fmt="{:.0f}")
compare(axs[2], "total_amount", "Average transaction nominal\n(Rp M)", scale=1e6)
fig.suptitle("Churner transactions less frequently and shorter durations, but the nominal per transaction almost same", x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 1]); plt.savefig("charts/2_perilaku_transaksi.png", dpi=150); plt.close()


[Transaction] average:
        n_trx  trx_span_days  total_amount
status                                    
Active    4.0           67.0    15289909.0
Churn     3.0           43.0    11597210.0


In [72]:
# 3. Frequency session and feature user

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.6]})
print("\n[Session] average:"); print(df.groupby("status")[["total_sessions", "active_days"]].mean().round(1))
compare(axs[0], "total_sessions", "Average Session")
compare(axs[1], "active_days", "Average Active Days")
feat = df.groupby("status")[["pct_browse", "pct_loan_apply", "pct_payment", "pct_promo_click"]].mean().T
feat.index = ["Browse", "Loan apply", "Payment", "Promo click"]; print("\n[Fitur] % .user:"); print(feat.round(1))
x = np.arange(4); w = 0.38
b1 = axs[2].bar(x - w / 2, feat["Active"], w, color=BLUE, label="Aktif")
b2 = axs[2].bar(x + w / 2, feat["Churn"], w, color=RED, label="Churn")
for bars in (b1, b2):
    for b in bars: axs[2].text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.0f}%", ha="center", va="bottom")
axs[2].set_xticks(x); axs[2].set_xticklabels(feat.index); axs[2].set_ylim(0, 35); axs[2].legend(frameon=False)
axs[2].set_title("Proportion of application feature usage")
fig.suptitle("All activity shows almost no significant different, Only promo click that have churner over active users", x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig("charts/3_sesi_dan_fitur.png", dpi=150); plt.close()


[Session] average:
        total_sessions  active_days
status                             
Active            35.5         11.2
Churn             35.8         11.4

[Fitur] % .user:
status       Active  Churn
Browse         25.0   22.8
Loan apply     25.4   24.8
Payment        25.6   24.3
Promo click    24.0   28.1


In [39]:
# 4. profil users: channel, city, age

df["age_group"] = pd.cut(df.age, [0, 29, 39, 49, 100], labels=["<30", "30-39", "40-49", "50+"])
fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
rate_bars(axs[0], "acquisition_channel", "Channel acquisition")
rate_bars(axs[1], "city", "City")
rate_bars(axs[2], "age_group", "Age group")
fig.suptitle("Churn rate evenly distributed in all channel, city, dan age (average = 30%)", x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig("charts/4_profil_user.png", dpi=150); plt.close()

In [43]:
df["segment"] = np.where(df.n_trx == 1, "High risk\n(only 1 transaction)", "Low risk\n(2+ transaction)")
seg = df.groupby("segment").agg(users=("is_churn", "size"), churned=("is_churn", "sum"))
seg["churn_rate"] = seg.churned / seg.users * 100
seg["pct_users"] = seg.users / seg.users.sum() * 100
seg["pct_churners"] = seg.churned / seg.churned.sum() * 100
seg = seg.loc[["High risk\n(only 1 transaction)", "Low risk\n(2+ transaction)"]]
print("\n[Segmentation]"); print(seg.round(1))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
bars = axs[0].bar(seg.index, seg.churn_rate, color=[RED, BLUE], width=0.55)
for b, v in zip(bars, seg.churn_rate): axs[0].text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}%", ha="center", va="bottom", fontweight="bold")
axs[0].set_ylim(0, 85); axs[0].set_ylabel("Churn rate (%)"); axs[0].set_title("Churn rate per segment")
x = np.arange(2); w = 0.38
b1 = axs[1].bar(x - w / 2, seg.pct_users, w, color=GREY, label="% from all users")
b2 = axs[1].bar(x + w / 2, seg.pct_churners, w, color=RED, label="% from all churners")
for bars in (b1, b2):
    for b in bars: axs[1].text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.0f}%", ha="center", va="bottom", fontweight="bold")
axs[1].set_xticks(x); axs[1].set_xticklabels(seg.index); axs[1].set_ylim(0, 100); axs[1].legend(frameon=False); axs[1].set_title("Proportion user vs proportion churner")
fig.suptitle("16% user (high-risk) contributed 36% churner with churn rate 69%", x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.92]); plt.savefig("charts/5_segmentasi_risiko.png", dpi=150); plt.close()


[Segmentation]
                                 users  churned  churn_rate  pct_users  \
segment                                                                  
High risk\n(only 1 transaction)     39       27        69.2       15.6   
Low risk\n(2+ transaction)         211       47        22.3       84.4   

                                 pct_churners  
segment                                        
High risk\n(only 1 transaction)          36.5  
Low risk\n(2+ transaction)               63.5  
